# Technical Indicator Features

## Overview

This notebook demonstrates the public functions in `technical_indicator_features` using compact synthetic indicator paths.
- Problem: price and volume observations need to be summarized into momentum, trend, and volatility features.
- Approach: provide FinanceToolkit-shaped synthetic indicator tables and combine RSI, moving average, MACD, MACD signal, and ATR into date-ticker rows.
- Technical Indicator Features: It produces a compact market-feature set for momentum, trend, and range volatility.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.data_preprocessing.technical_indicator_features import (
    collect_technical_indicator_features,
)

## Collect Technical Indicator Features

This cell defines a synthetic FinanceToolkit technical provider and collects feature rows.
- The indicator paths describe a gradual positive trend without an extreme overbought RSI reading.
- MACD rises above its signal line while ATR stays positive, giving an interpretable momentum and volatility example.
- The synthetic provider follows the public `toolkit.technicals` interface used by the feature module.

In [ ]:
dates = pd.date_range("2025-01-02", periods=6, freq="B", name="date")

class SyntheticTechnicals:
    def __init__(self):
        self.values = {
            "get_relative_strength_index": [48, 51, 54, 57, 60, 62],
            "get_moving_average": [100, 101, 102, 103, 104, 105],
            "get_average_true_range": [1.8, 1.7, 1.6, 1.7, 1.8, 1.9],
        }

    def get_moving_average_convergence_divergence(self, **kwargs):
        macd = self._frame([-0.4, -0.1, 0.2, 0.5, 0.8, 1.0])
        signal = self._frame([-0.3, -0.2, 0.0, 0.3, 0.5, 0.7])
        return macd, signal

    def _frame(self, values):
        return pd.DataFrame({"AAPL": values}, index=dates)

    def __getattr__(self, name):
        def get_indicator(**kwargs):
            return self._frame(self.values[name])
        return get_indicator

class SyntheticToolkit:
    technicals = SyntheticTechnicals()

technical_features = collect_technical_indicator_features(SyntheticToolkit())

assert technical_features["average_true_range"].gt(0).all()
assert technical_features.loc[5, "macd"] > technical_features.loc[5, "macd_signal"]
assert technical_features["relative_strength_index"].between(0, 100).all()
display(technical_features)

This cell visualizes the synthetic momentum, trend, and volatility features.
- RSI remains between 0 and 100 and rises without reaching the common overbought threshold of 70.
- MACD finishing above its signal line is consistent with the positive trend in the moving average.
- ATR remains positive and modestly increases near the end of the sample.

In [ ]:
plot_data = technical_features.set_index("date")
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
plot_data[["relative_strength_index", "average_true_range"]].plot(ax=axes[0], marker="o")
axes[0].axhline(70, color="tab:red", linestyle="--", linewidth=1)
axes[0].set_title("Momentum And Range Volatility")
axes[0].set_ylabel("Indicator value")
plot_data[["macd", "macd_signal"]].plot(ax=axes[1], marker="o")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("MACD And Signal")
axes[1].set_ylabel("MACD")
plt.tight_layout()
plt.show()